In [ ]:
# !uv pip install numpy quantities matplotlib scipy neo nixio pyocclient tqdm

<a target="_blank" href="https://colab.research.google.com/github/ANDA-NI-2026/NI-Day2-Data-Models/blob/main/03_working_with_datasets_in_neo/03_exercises.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Neo (Part 2): Organizing Electrophysiology Data

In the first session you worked with the Neo data objects `AnalogSignal`, `SpikeTrain`, `Event`, `Epoch`. In this session, you will learn how to organise these data objects into containers called `Block` and `Segment` for a complete, trial-based dataset, much like one you would obtain by loading a recording from disk.

You will also get practice exploring the contents of a dataset, select subsets of the data using metadata, cut the recording into per-trial segments with Neo's utility functions, and finally transform and visualise a signal.

## Setup

### Import Libraries

In [ ]:
import neo
import numpy as np
import quantities as pq
import matplotlib.pyplot as plt

from neo.utils import add_epoch, cut_segment_by_epoch, get_events
from neo.core.filters import IsNot, IsIn, GreaterThan, LessThan, GreaterThanOrEquals, LessThanOrEquals, InRange

### Generate Data

Please run the following cell, which will generate a data file that we will use later.

In [ ]:
from pathlib import Path

rng = np.random.default_rng(0)

# A 96-channel LFP-like signal sampled at 1 kHz for 10 s.
lfp = neo.AnalogSignal(rng.normal(size=(10000, 96)), units='uV',
                       sampling_rate=1000*pq.Hz, t_start=0*pq.s,
                       name='LFP')
lfp.array_annotate(channel_names=np.array([f'chan{i+1}' for i in range(96)]))

# A smaller 6-channel signal sampled at 30 kHz for 10 s.
hires = neo.AnalogSignal(rng.normal(size=(300000, 6)), units='uV',
                         sampling_rate=30000*pq.Hz, t_start=0*pq.s,
                         name='Subsample unprocessed signal')
hires.array_annotate(channel_names=np.array([f'ainp{i+1}' for i in range(6)]))

# A handful of spike trains, each labelled as single-unit, multi-unit or noise.
unit_types = ['sua', 'mua', 'noise', 'sua', 'mua', 'sua', 'noise', 'sua']
spiketrains = []
for i, unit_type in enumerate(unit_types):
    times = np.sort(rng.uniform(0, 10, size=rng.integers(50, 150)))
    st = neo.SpikeTrain(times*pq.s, t_start=0*pq.s, t_stop=10*pq.s, name=f'unit {i}')
    st.annotate(unit_type=unit_type)
    spiketrains.append(st)

# Trial-start events, alternating correct and error, every 3 s.
ts_times = np.arange(1, 10, 3) * pq.s
performance = np.array(['correct_trial' if i % 2 == 0 else 'error_trial'
                        for i in range(len(ts_times))])
events = neo.Event(ts_times, labels=np.array(['TS-ON'] * len(ts_times)),
                   name='TrialEvents')
events.array_annotate(performance_in_trial_str=performance)

# Assemble everything into a Block with one Segment.
segment = neo.Segment(name='recording')
segment.analogsignals.extend([lfp, hires])
segment.spiketrains.extend(spiketrains)
segment.events.append(events)
block = neo.Block(name='dataset')
block.segments.append(segment)

# Write the block to a NIX file that the notebook reads back in.
out_path = Path('dataset.nix')
out_path.unlink(missing_ok=True)
with neo.NixIO(str(out_path), mode='ow') as io:
    io.write_block(block)
print(f'Wrote dataset to {out_path}')

Wrote dataset to dataset.nix


## Organizing Data: `Block` and `Segment`

### Background

A real recording often contains several of the data objects `AnalogSignal`, `SpikeTrain`, `Event`, `Epoch`, and Neo organises them in a simple two-level hierarchy:

- A **`Segment`** holds all data objects that share a common clock — for example everything recorded during a single trial or a single continuous recording. Its data is stored in lists: `segment.analogsignals`, `segment.spiketrains`, `segment.events`, and `segment.epochs`.
- A **`Block`** is the top-level container that holds everything; its `block.segments` list holds one or more segments.

Once data is organised this way, the `filter` method lets you search a block (or segment) for all objects of a given type or with a given annotation, without walking the hierarchy by hand.

### Exercises

In this section you will build a `Block` containing a `Segment`, add data objects to it, navigate the hierarchy, and search it with `filter`.

| Code | Description |
| :-- | :-- |
| `neo.Block(name='my block')` | Create a block (the top-level container). |
| `neo.Segment(name='trial 1')` | Create a segment (a group of data with a shared clock). |
| `block.segments.append(seg)` | Add a segment to a block. |
| `neo.AnalogSignal(np.random.random((100, 4)), units='uV', sampling_rate=1000*pq.Hz)` | Create an `AnalogSignal` from a NumPy array, units, and a sampling rate. |
| `seg.analogsignals.append(sig)` | Add an `AnalogSignal` to a segment. |
| `neo.SpikeTrain([0.1, 0.4, 0.8], units='s', t_start=0*pq.s, t_stop=1*pq.s)` | Create a `SpikeTrain` from a list of spike times. |
| `seg.spiketrains.append(st)` | Add a `SpikeTrain` to a segment. |
| `obj.annotate(key=value)` | Attach a named annotation to any Neo object. |
| `print(seg.spiketrains)` | Display the list of spike trains in a segment. |
| `block.filter(objects=neo.AnalogSignal)` | Find all analog signals anywhere in the block. |
| `block.filter(objects=neo.SpikeTrain)` | Find all spike trains anywhere in the block. |

**Example**: Create a block named `example block` containing one segment named `example segment`. Dissplay the block.

In [ ]:
block = neo.Block(name='my block')
seg = neo.Segment(name='trial 1')
block.segments.append(seg)
block

Block with [<neo.core.segment.Segment object at 0x719efebfa0f0>] segments
name: 'my block'
# segments (N=[<neo.core.segment.Segment object at 0x719efebfa0f0>])
0: Segment with  name: 'trial 1' # analogsignals (N=[])

Add an `AnalogSignal` with `100` samples in `4` different channels with units `uV` and sampling rate `1000 Hz` to that segment. Display the block container to check that the segment now has the `AnalogSignal` object.

In [ ]:
sig = neo.AnalogSignal(np.random.random((100, 4)), units='uV', sampling_rate=1000*pq.Hz)
seg.analogsignals.append(sig)
block

Block with [<neo.core.segment.Segment object at 0x719efebfa0f0>] segments
name: 'my block'
# segments (N=[<neo.core.segment.Segment object at 0x719efebfa0f0>])
0: Segment with [<AnalogSignal(array([[0.96716973, 0.78566427, 0.32522704, 0.53868382],
       [0.53183792, 0.16296368, 0.63590246, 0.63462643],
       [0.92417252, 0.09243477, 0.67039178, 0.27857824],
       [0.28564974, 0.31629538, 0.17615652, 0.70590437],
       [0.79088043, 0.00824679, 0.30761956, 0.46385119],
       [0.97849895, 0.29309182, 0.23327034, 0.1452885 ],
       [0.21974271, 0.29429664, 0.20824123, 0.51386939],
       [0.6911154 , 0.36907825, 0.45129478, 0.24461729],
       [0.8439489 , 0.97350243, 0.24846843, 0.79094075],
       [0.00317768, 0.33423438, 0.8968902 , 0.1271694 ],
       [0.50656325, 0.2431994 , 0.50720428, 0.83907496],
       [0.88153646, 0.30547195, 0.16690663, 0.49627213],
       [0.06195734, 0.79330984, 0.48105828, 0.02209779],
       [0.82680942, 0.02662455, 0.14294113, 0.61799224],
       [0.4

**Exercise**: Create a new block named `'block1'` and store it in a variable `block1`. Add one segment named `'trial 1'` to it. Display `block1`.

Add an `AnalogSignal` with `20` samples in `8` different channels with units `mV` and sampling rate `30 kHz` to that segment. Display the block container to check that the segment has the `AnalogSignal` object.

**Exercise**: Add a `SpikeTrain` object containing the timestamps `[0.1, 0.4, 0.8]` to the segment's `spiketrains` list, annotated with `unit_id=0`. Display the spiketrains of the segment using `print`.

**Exercise**: Add another `SpikeTrain` object containing the timestamps `[0.2, 0.6]` to the segment's `spiketrains` list, annotated with `unit_id=1`. Print the segment spiketrains to see that it now contains both spike trains.

**Exercise**: Use `filter` to get a list of all `AnalogSignal` objects in the block.

**Exercise**: Use `filter` to get a list of all `SpikeTrain` objects in the block, and report how many there are.

## Exploring a Dataset

### Background

When you load a recording, the first thing to do is find out what it contains: how many continuous recording parts (segments) there are, how many channels were recorded and at what sampling rates, and how many spike trains are present. The `Block` and `Segment` objects expose this through their lists (`block.segments`, `segment.analogsignals`, `segment.spiketrains`, `segment.events`), and the `filter` method lets you select objects by type or by annotation.

### Exercises

In this section you will inspect the structure of `block`, read off the properties of its signals, and use annotations to select a meaningful subset of the spike trains.

| Code | Description |
| :-- | :-- |
| `block` | Print a summary of a block and its contents. |
| `block.segments` | The list of segments in the block. |
| `len(block.segments)` | The number of segments in the block. |
| `segment = block.segments[0]` | Access the first (here, only) segment. |
| `segment.analogsignals` | The list of `AnalogSignal` objects in a segment. |
| `sig.name`, `sig.shape`, `sig.sampling_rate` | Properties of an `AnalogSignal`. |
| `segment.spiketrains` | The list of `SpikeTrain` objects in a segment. |
| `len(segment.spiketrains)` | The number of spike trains in a segment. |
| `block.filter(objects=neo.SpikeTrain, targdict={'unit_type': 'sua'})` | Find all spike trains with a given annotation. |

**The Dataset**

Run the cell below to load the dataset used for the rest of this notebook from the NIX file in the `data` folder. It contains a single `Block` with one `Segment` that contains:

- two `AnalogSignal` objects, both spanning 10 s: a 96-channel LFP-like signal sampled at 1 kHz, and a smaller 6-channel signal sampled at 30 kHz, each with per-channel `channel_names`;
- a set of `SpikeTrain` objects, each annotated as `'sua'`, `'mua'`, or `'noise'`;
- an `Event` object marking trial-start (`TS-ON`) times, with a per-event annotation recording whether each trial was a `'correct_trial'` or an `'error_trial'`.

This is the kind of structure you obtain when loading a real recording from disk.

In [ ]:
with neo.NixIO('dataset.nix', mode='ro') as io:
    block = io.read_block()
block

Block with [<neo.core.segment.Segment object at 0x719efe98c8d0>] segments
name: 'dataset'
annotations: {'nix_name': 'neo.block.563d20de96454358ab5e2b434ae63016'}
rec_datetime: datetime.datetime(2026, 6, 23, 15, 43, 46)
# segments (N=[<neo.core.segment.Segment object at 0x719efe98c8d0>])
0: Segment with [<AnalogSignal(array([[ 0.12573022, -0.13210486,  0.64042265, ..., -0.73548329,
         0.24978537,  1.03145308],
       [ 0.16100958, -0.58552882, -1.34121971, ..., -0.47433299,
        -1.94426498, -1.3077532 ],
       [ 1.08683078, -0.05060406, -0.28312507, ...,  0.11566183,
        -1.07054441, -1.0026843 ],
       ...,
       [ 0.55947905,  0.18952103, -0.67187467, ..., -0.05874888,
        -0.11022754, -1.18243894],
       [-0.73628695, -1.45294563,  0.08538314, ..., -0.25738861,
         2.05335138,  2.77651512],
       [ 1.06179533, -0.97950888, -0.9572926 , ..., -1.33537531,
        -0.5644885 ,  1.39068265]], shape=(10000, 96)) * uV, [0.0 s, 10.0 s], sampling rate: 1000.0 Hz)>

**Exercise**: How many segments does the block contain? Hint: use `len`.

**Exercise**: Assign the first segment to a variable `segment`. How many `AnalogSignal` objects does the segment contain?

**Exercise**: For the first `AnalogSignal` in the segment, print its name, shape and sampling rate.

**Exercise**: For the second `AnalogSignal` in the segment, print its name, shape and sampling rate. Which of the two signals has the higher temporal resolution?

**Exercise**: How many `SpikeTrain` objects does the segment contain?

**Exercise**: Use `filter` to get all spike trains annotated as single-unit activity (`unit_type` equal to `'sua'`), and report how many there are.

**Exercise**: Use `filter` to get all spike trains annotated as multi-unit activity (`unit_type` equal to `'mua'`), and report how many there are.

**Exercise**: Use `filter` to get all spike trains annotated as noise (`unit_type` equal to `'noise'`). How many noise spike trains does the block contain?

The exact-value `targdict` matching used above works well for equality tests, but Neo also supports `FilterCondition` objects from `neo.core.filters` that express richer queries — inequality tests and set membership — directly inside `block.filter()` without needing a separate list comprehension.

| Condition | Description |
| --- | --- |
| `IsNot(x)` | annotation **≠** x |
| `IsIn([a, b])` | annotation is one of a, b |

**Exercise**: Use `block.filter()` with `IsNot('noise')` on the `unit_type` annotation to retrieve all spike trains that are **not** noise in a single call. Print how many are returned, and compare with the sum of the SUA and MUA counts found earlier.

**Exercise**: Use `IsIn(['sua', 'mua'])` to retrieve all spike trains whose `unit_type` is either `'sua'` or `'mua'` in one call. Confirm the result is the same as the `IsNot('noise')` query above.

The equality and membership conditions used above (`IsNot`, `IsIn`) are useful for string annotations. When annotations carry **numeric** values, Neo's `filter` also supports inequality and range conditions from `neo.core.filters`:

| Condition | Description |
| --- | --- |
| `GreaterThan(x)` | annotation value **>** x |
| `LessThan(x)` | annotation value **<** x |
| `GreaterThanOrEquals(x)` | annotation value **≥** x |
| `LessThanOrEquals(x)` | annotation value **≤** x |
| `InRange(lo, hi)` | lo **≤** value **≤** hi |

These conditions are passed as values in `targdict`, e.g. `block.filter(objects=neo.SpikeTrain, targdict={'unit_id': GreaterThan(3)})`.

**Exercise**: Add four more empty `SpikeTrain` objects (`t_stop=1*pq.s`) to `block1`'s segment (which already has two spike trains attached from a previous exercise), annotated with `unit_id` from 2 to 5. Then use `GreaterThan(3)` to retrieve all spike trains in `block1` whose `unit_id` is strictly greater than 3. Print the `unit_id` annotation of each returned spike train.

**Exercise**: Use `InRange(1, 3)` to retrieve all spike trains in `block1` whose `unit_id` lies between 1 and 3 (bounds inclusive). Print the matching `unit_id` values and confirm the count.

## Selecting and Masking Data

### Background

Analysis usually focuses on a subset of the data: certain channels, certain trials, or a certain time window. Neo gives you several complementary ways to slice data out of an object:

- **Boolean masks on array annotations** select the events (or channels) whose metadata satisfies a condition. A mask is a boolean array you build by comparing an array annotation to a value. Indexing an `Event` with a mask returns a new `Event` with just those time points.
- **Column slicing** (`sig[:, indices]`) selects channels of an `AnalogSignal`.
- **`time_slice(t_start, t_stop)`** returns a new object containing only the data in a time window, and works on signals, spike trains, and whole segments.

### Exercises

In this section you will select events from correct trials with a mask, pick out a range of channels, and extract a time window of a signal.

| Code | Description |
| :-- | :-- |
| `segment.events[0]` | Access the first `Event` object in a segment. |
| `ev.array_annotations['performance_in_trial_str']` | Read a per-event array annotation. |
| `mask = ev.array_annotations['key'] == 'value'` | Build a boolean mask from a condition. |
| `ev[mask]` | Select the events where the mask is `True`. |
| `len(events)` | The number of time points in an `Event`. |
| `sig[:, 10:20]` | Select a range of channels (columns) of a signal. |
| `sig.shape` | The shape of the underlying data array. |
| `sig.array_annotations['channel_names']` | Read the names of the channels. |
| `sig.time_slice(5*pq.s, 6*pq.s)` | Extract the data between two time points. |

**Example**: The segment's `Event` object marks trial starts. Build a mask selecting only the events from correct trials, and use it to create a new `Event` containing just those time points. `performance_in_trial_str` is an array annotation that shows, for each individual event contained within the object, a string indicating the performance of the trial during which the event was recorded.

In [ ]:
events = segment.events[0]
mask = events.array_annotations['performance_in_trial_str'] == 'correct_trial'
correct_events = events[mask]
correct_events

Event containing 2 events with labels; time units s; datatype float64 
name: 'TrialEvents'
annotations: {'nix_name': 'neo.event.65634012877f48fcb749cd23d72aacb3'}

**Exercise**: How many trial-start events were there in total, and how many came from correct trials? Hint: use `len`.

**Exercise**: Select the channels at indices 10 to 19 (inclusive) of the LFP signal (the first `AnalogSignal` of the segment) and assign it to a variable named `lfp_subset`. Check the shape of the data in this subset of LFP channels.

**Exercise**: Read the `channel_names` array annotation of `lfp_subset` to confirm which channels you selected.

**Exercise**: Use `time_slice` to extract the data of `lfp` between 5 and 6 seconds, and check the shape of the result (it should contain 1 second of data at 1 kHz from 96 channels).

**Exercise**: Use `time_slice` to extract the data of `lfp_subset` between 1 and 5 seconds, and check the shape of the result (it should contain 4 seconds of data at 1 kHz from 10 chanels).

Beyond building boolean masks by hand, `neo.utils.get_events()` provides a convenience function that filters an event by any combination of its array annotations in a single call, returning a list of matching `Event` objects.

**Exercise**: Use `neo.utils.get_events(segment, performance_in_trial_str='correct_trial')` to retrieve the correct-trial events in one call. Print how many events are returned and compare with the mask approach used above.

## Epochs and Cutting Trials

### Background

A common task is to align data to an experimental event — for example, to look at every signal in a window around each trial start. Neo provides two utility functions in `neo.utils` that make this straightforward:

- **`add_epoch`** builds an `Epoch` object spanning a fixed window around a set of events. You give it the segment, the events to anchor on (`event1`), and the window edges relative to each event (`pre` and `post`). Leaving `event2=None` cuts a fixed window around single events rather than between pairs of events.
- **`cut_segment_by_epoch`** uses such an epoch to slice a segment into a list of
  new segments, one per epoch. With `reset_time=True`, every resulting segment starts at time 0, so trials are aligned and directly comparable.

### Exercises

In this section you will build analysis epochs around the correct-trial starts and cut the recording into one aligned segment per trial.

| Code | Description |
| :-- | :-- |
| `add_epoch(segment, event1=ev, event2=None, pre=-100*pq.ms, post=500*pq.ms, attach_result=False, name='analysis_epochs')` | Build an epoch in a window around each event. |
| `len(epoch)` | The number of epochs created. |
| `epoch.durations` | The duration of each epoch. |
| `cut_segment_by_epoch(segment, epoch, reset_time=True)` | Cut a segment into one segment per epoch, aligned to time 0. |
| `seg.analogsignals[0].t_start` | The start time of a signal in a cut segment. |

Run the cell below to rebuild the correct-trial start events used in this section.


**Example**: Build an `Epoch` spanning from 100 ms before to 500 ms after each correct-trial start. The epoch is not attached to the segment (`attach_result=False`).

In [ ]:
epoch = add_epoch(segment,
                  event1=correct_events, event2=None,
                  pre=-100*pq.ms, post=500*pq.ms,
                  attach_result=False,
                  name='analysis_epochs')
epoch

Epoch containing 2 epochs with labels; time units s; datatype float64 
name: 'analysis_epochs'

**Exercise**: How many epochs were created, and what is the duration of each? (There should be one epoch per correct-trial event, and each should last 600 ms.)

**Exercise**: Use `cut_segment_by_epoch` to cut the segment into one segment per epoch, aligning every trial to time 0 with `reset_time=True`. Store the result in a variable `trial_segments`.

**Exercise**: Confirm the alignment worked by checking the `t_start` of the first `AnalogSignal` in each trial segment — they should all start at the same time.

**Exercise**: Verify that per-channel annotations survived the cut by printing the `array_annotations` of the first `AnalogSignal` in the first trial segment.

**Exercise**: Compare spike counts across trial segments for the first spike train: print `len(seg.spiketrains[0])` for each trial segment as a list. Do the counts differ across trials? Why would you expect that?

## Transforming and Visualizing

### Background

Because Neo objects are unit-aware NumPy arrays, they support a range of transformations that preserve their metadata, and they integrate cleanly with plotting libraries such as Matplotlib. Useful operations include:

- **`rescale(unit)`** — convert the data to a different but compatible physical
  unit (e.g. microvolts to millivolts).
- **`magnitude`** — strip the units and return a plain NumPy array, which is what
  you pass to plotting functions.
- **`downsample(factor)`** — return a new signal with the sampling rate reduced
  by an integer factor.

For plotting, `sig.times` gives the x-axis values and `sig.magnitude` gives the y-axis values, while `array_annotations` supply per-channel labels for the legend.

### Exercises

In this section you will rescale and downsample a signal, and produce a labelled plot of a few channels.

| Code | Description |
| :-- | :-- |
| `sig.time_slice(0*pq.s, 1*pq.s)` | Extract a time window of the signal. |
| `sig.rescale(pq.mV)` | Convert the data to another compatible unit. |
| `sig.magnitude` | The data as a plain NumPy array, without units. |
| `sig.downsample(10)` | Reduce the sampling rate by an integer factor. |
| `sig.sampling_rate` | The sampling rate of the signal. |
| `sig.times` | The time stamps of the samples (with units). |
| `sig.dimensionality`, `sig.times.dimensionality` | The units of the signal and of its time stamps, useful for axis labels. |
| `sig.array_annotations['channel_names']` | Per-channel labels for a legend. |
| `plt.plot(x, y, label='my_label')` | Plot a line and give it a legend label. |

**Example**: Take the first second of the LFP signal and rescale it from microvolts to millivolts.

In [ ]:
lfp = segment.analogsignals[0]
lfp_1s = lfp.time_slice(0*pq.s, 1*pq.s)
lfp_mv = lfp_1s.rescale(pq.mV)
lfp_mv

AnalogSignal with 96 channels of length 1000; units mV; datatype float64
name: 'LFP'
annotations: {'nix_name': 'neo.analogsignal.8a729323193c4f18a9f43821db80a09e'}
sampling rate: 1000.0 Hz
time: 0.0 s to 1.0 s

**Exercise**: Get the underlying data of `lfp_1s` as a plain NumPy array (without units) and check its type and shape.

**Exercise**: Downsample `lfp` by a factor of 10 and confirm the new sampling rate is 100 Hz.

**Exercise**: Plot the first second of the channels at indices 10 to 14 of the LFP signal. Use `sig.times` for the x-axis and `sig.magnitude` for the y-axis, and label each line with its name from the `channel_names` array annotation. Add axis labels, a title, and a legend. For the axis labels, do not hard code the units, as you can obtain them from the object you are plotting using the `.dimensionality` attribute (see code reference).

## Visualizing Trial Data (Bonus)

Now that the recording is cut into aligned trial segments, we can produce the most common visualization in systems neuroscience: a **raster plot**, which shows the spike times of one unit stacked across all trials. We can also compute a **trial-averaged signal** (i.e., the mean LFP across trials).

**Exercise**: Collect the first spike train (`spiketrains[0]`) from every trial segment and plot a raster. For each trial `i`, use `ax.plot(st.rescale('ms').magnitude, [i]*len(st), '|k', markersize=6)`. Add axis labels, a title, and a vertical dashed red line at `x=0` to mark the reference event.

**Exercise**: Use `time_slice` to extract only the first 300 ms after the reference event (`0*pq.ms` to `300*pq.ms`) from the first spike train of the first trial segment. Print `len` before and after slicing.

**Exercise**: Compute and plot the trial-averaged LFP of channel 0. Collect `seg.analogsignals[0][:, 0].magnitude` from every trial segment, compute `np.mean(..., axis=0)`, and plot it against the time axis taken from the first trial segment. Add axis labels and a vertical dashed red line at `x=0` marking the reference event.

> **Note**: This section uses `nixio`, which requires a compiled HDF5 library. It runs correctly with the course's pixi environment but **cannot run in JupyterLite** (browser-based execution). Run these cells with a local kernel.

## From Neo to NIX: Storing Neuroscience Data

**NIX** is an HDF5-based file format purpose-built for neuroscience data. Its central idea is an explicit *object model* that keeps data and its description inseparable:

- A **`File`** contains one or more **`Block`** objects, each representing a recording session or dataset. (Note: This is **not** a Neo Block!)
- Each `Block` contains **`DataArray`** objects. A `DataArray` wraps a numpy array and stores its physical `unit` and a human-readable `label`.
- Every axis of a `DataArray` is described by a **dimension descriptor**: a `SampledDimension` for regularly sampled axes (e.g. a time axis at 1 kHz), or a `SetDimension` for categorical axes (e.g. channel names).

This object model maps naturally onto Neo: when `neo.io.NixIO` writes a Neo `Block` to disk it creates a NIX file containing a NIX `Block`, with every `AnalogSignal` channel stored as its own `DataArray` with proper dimension descriptors. Neo's UUID-based internal naming is designed for round-tripping; when you write your own analysis results to NIX you are free to use clean, human-readable names.

The exercises below first introduce the raw `nixio` API, then show how to peek inside a Neo-generated NIX file and how to store analysis results with descriptive metadata. As an outlook, besides storing Neo objects in the NIX file format, you can store additional, non-electrophysiology data in the NIX file using `nixio`, such as behavioral data or analysis results.

| Code | Description |
| --- | --- |
| `nixio.File.open('f.nix', nixio.FileMode.Overwrite)` | Create a new file (overwrites if it exists) |
| `nixio.File.open('f.nix', nixio.FileMode.ReadOnly)` | Open an existing file for reading |
| `nixio.File.open('f.nix', nixio.FileMode.ReadWrite)` | Open an existing file for reading and writing |
| `with nixio.File.open(...) as nf:` | Context manager — always use this form to ensure the file is closed |
| `nf.blocks` | List of all blocks in the file |
| `blk = nf.create_block('name', 'type')` | Create a `Block` inside the file |
| `da = blk.create_data_array('name', 'type', data=arr, unit='mV', label='voltage')` | Create a `DataArray` from a numpy array |
| `blk.data_arrays[0]` | Access a `DataArray` by index |
| `da[:]` | Read all data back as a numpy array |
| `da.shape`, `da.unit`, `da.label` | Properties of a `DataArray` |
| `da.append_sampled_dimension(dt, unit='s', label='time')` | Attach a regularly-sampled axis (e.g. a time axis) |
| `da.append_set_dimension(labels=['Fz', 'Cz', 'Pz'])` | Attach a categorical axis with explicit labels |
| `da.dimensions[0].dimension_type` | Type of the first axis descriptor |
| `da.dimensions[0].sampling_interval` | Sampling interval of a `SampledDimension` |
| `da.dimensions[1].labels` | Category labels of a `SetDimension` |

In [ ]:
import nixio

**Exercise**: Create a file `'recording.nix'` in `Overwrite` mode using the context manager. Inside the `with` block, print `nf.blocks` to confirm the file starts empty, then create a block named `'session'` with type `'nix.session'`. Print `nf.blocks` again to confirm the block was added.

**Exercise**: Reopen `'recording.nix'` in `ReadWrite` mode. Retrieve the block (`nf.blocks[0]`). Create a `DataArray` named `'voltage'` from the 1D sine-wave signal defined in the cell below, with `unit='mV'` and `label='membrane voltage'`. Append a `SampledDimension` with `sampling_interval=0.001`, `unit='s'`, and `label='time'`. Print `da.shape`.

**Exercise**: Reopen `'recording.nix'` in `ReadOnly` mode. Access the first block and first `DataArray`. Print `da.name`, `da.unit`, `da.label`, and `da.shape`. Then print the first 5 values with `da[:5]`.

**Exercise**: Reopen `'recording.nix'` in `ReadOnly` mode. Read `da.dimensions[0].dimension_type` and `da.dimensions[0].sampling_interval` for the voltage `DataArray`. How does the sampling interval relate to the `dt` used when creating the signal?

**Exercise**: Reopen `'recording.nix'` in `ReadWrite` mode. Create a second `DataArray` named `'eeg'` from the 2D EEG-like array defined in the cell below (shape `(1000, 3)`, unit `'uV'`, label `'EEG signal'`). Attach a `SampledDimension` (1 ms interval, unit `'s'`, label `'time'`) for the first axis and a `SetDimension` with `labels=['Fz', 'Cz', 'Pz']` for the second. Print `da2.shape`.

**Exercise**: Reopen `'recording.nix'` in `ReadOnly` mode. Access the second `DataArray` (`blk.data_arrays[1]`). Print the `dimension_type` and `sampling_interval` of the first dimension, and the `dimension_type` and `labels` of the second dimension. Confirm the channel names survived the round trip.

**Exercise**: Open `data/dataset.nix'` in `ReadOnly` mode with raw `nixio`. Access the first block with `nf.blocks[0]`. Print the block's `name`, the total number of `DataArrays`, and the number of `MultiTags`. Then print the `name` of `blk.data_arrays[0]`. What naming scheme does NixIO use?

NixIO uses UUID-based names (`neo.analogsignal.<hash>.<channel_index>`) and stores one `DataArray` per channel. This internal convention is designed for round-tripping through Neo — it is not meant to be navigated directly with `nixio`. When *you* create a NIX file for your own results, you can choose clean, descriptive names and store multi-channel data as a single 2-D `DataArray` with a `SetDimension` for the channel axis.

The cells below rebuild the `trial_segments` and `mean_lfp` variables used in the previous section, so this section can be run independently.

In [ ]:
# reload block
with neo.NixIO('dataset.nix', mode='ro') as io:
    block = io.read_block()
segment = block.segments[0]

# rebuild trial cuts (correct events, -100 ms to +500 ms window)
events        = segment.events[0]
correct_mask  = events.array_annotations['performance_in_trial_str'] == 'correct_trial'
correct_ev    = events[correct_mask]
epoch         = add_epoch(segment, event1=correct_ev,
                          pre=-100*pq.ms, post=500*pq.ms,
                          attach_result=False, name='analysis_epochs')
trial_segments = cut_segment_by_epoch(segment, epoch, reset_time=True)

# compute trial-averaged LFP (channel 0)
lfp_trials = [seg.analogsignals[0][:, 0].magnitude for seg in trial_segments]
mean_lfp   = np.mean(lfp_trials, axis=0)
n_trials   = len(trial_segments)
dt_s       = float(trial_segments[0].analogsignals[0].sampling_period
                   .rescale('s').magnitude)
print(f'mean_lfp shape: {mean_lfp.shape}, dt={dt_s} s, n_trials={n_trials}')

mean_lfp shape: (600, 1), dt=0.001 s, n_trials=2


**Exercise**: Create a file `'analysis_results.nix'` in `Overwrite` mode. Inside it, create a block named `'analysis'`. Store `mean_lfp` as a `DataArray` named `'mean LFP'` with `unit='uV'` and `label='LFP channel 0'`. Append a `SampledDimension` using `dt_s` as the sampling interval (unit `'s'`, label `'time'`). Print `da.shape`.

**Exercise**: Reopen `'analysis_results.nix'` in `ReadWrite` mode. Create a metadata `Section` named `'analysis info'` with type `'nix.metadata'`. Add three properties: `'analysis_type'` (`'trial-averaged LFP'`), `'n_trials'` (the integer `n_trials`), and `'source_file'` (`'data/dataset.nix'`). Link the section to the `DataArray` via `da.metadata = sec`.

**Exercise**: Reopen `'analysis_results.nix'` in `ReadOnly` mode. Read back the `DataArray` and print its `name`, `shape`, `unit`, and the sampling interval of its first dimension. Then iterate over `da.metadata.props` and print each property name and its first value.